#### **Project:** PBL 4.프로덕트 분석
#### **분석자 :** 데분 4기 박의진
#### **분석일자 :** 2026.02.05
##### **데이터셋 :** [스타벅스 앱 고객 데이터](https://www.kaggle.com/datasets/blacktile/starbucks-app-customer-reward-program-data)
##### **참고 :** 
- 중심 테이블 : transcript.json (행동 로그)
- 고객 정보 : profile.json
- 오퍼 정보 : portfolio.json



---
#### **프로젝트 가이드라인**
##### 프로젝트 목표

- **고객 세분화**
    - RFM 분석 및 K-Means Clustering을 통해 고객을 유의미한 그룹으로 분류하고 페르소나를 정의합니다.
- **리텐션 및 전환 분석**
    - **코호트 분석**을 통해 고객 유지율을 파악하고, **퍼널 분석**을 통해 구매 여정의 병목 구간을 발굴합니다.
- **행동 예측**
    - 머신러닝 알고리즘을 활용하여 고객의 이탈 가능성(Churn)을 예측합니다.
- **전략 수립**
    - 분석 결과를 바탕으로 마케팅, CRM, 프로덕트 개선 등 실질적인 비즈니스 전략을 제안합니다.

##### 1️⃣ 문제 정의

- 우리 서비스의 현황은 어떠한가? (매출 추이, 고객 수 등)
- 우리가 집중해야 할 핵심 고객군은 누구인가?
- **고객은 언제, 어디서 이탈하는가?** (리텐션이 떨어지는 시점, 구매 포기 구간 등)
- **최종 목표:** "누구에게, 언제, 무엇을 제안해야 매출과 재구매율이 오를까?"에 대한 답을 찾아봅니다.

##### 2️⃣ 분석 및 모델링

*데이터셋의 특성에 따라 분석 기법의 구체적인 적용 방식은 달라질 수 있습니다.*

**[필수] 핵심 분석 기법**

- **EDA & 전처리:** 데이터 구조 파악, 결측치/이상치 처리, 파생 변수 생성.
- **고객 세분화 (Clustering):** RFM 지표 산출 및 K-Means 군집화 수행.
- **코호트 분석 (Cohort Analysis):**
    - 가입 월(또는 첫 구매 월) 별로 그룹을 나누어 `Time Cohort` 분석 수행.
    - 히트맵(Heatmap)을 통해 기간 경과에 따른 재구매율(Retention Rate) 변화 시각화.
- **퍼널 분석 (Funnel Analysis):**
    - 사용자의 주요 행동 흐름(예: `메인 접속` -> `상품 열람` -> `장바구니` -> `구매`) 정의.
    - 단계별 전환율(Conversion Rate) 및 이탈률(Drop-off Rate) 계산.
- **이탈 예측 (Churn Prediction):**
    - 이탈 정의 후 Classification 모델 학습 (Feature Importance 분석 포함).

**[선택] 도전 과제**

*팀의 역량에 따라 아래 기법 중 하나를 추가로 시도해 볼 수 있습니다.*

- **장바구니 분석 (Association Rules):** `Apriori` 알고리즘 등을 활용하여 "함께 구매되는 상품 조합" 발견.
- **시계열 매출 예측 (Time Series):** `Prophet` 라이브러리 등을 활용하여 향후 매출 추세 예측.

##### 3️⃣ **인사이트 제안**

- 단순한 수치 나열이 아닌, **"그래서 무엇을 해야 하는가?"**에 대한 구체적인 액션 아이템을 도출합니다.
    - *예: "코호트 분석 결과, 가입 3개월 차에 이탈이 급증하므로 2개월 차 고객 대상 프로모션 진행"*
    - *예: "퍼널 분석 결과, 장바구니에서 결제로 넘어가는 전환율이 낮으므로 결제 페이지 UI 개선 제안"*
 

---
# STEP 1. DATA Understanding
## 목적 
데이터 구조를 정밀 점검하고 넘어가기 : 이 데이터로 무엇을 어떻게 분석할 수 있는가 감을 잡기 위한 단계임.
### 진행 프로세스 및 결과 요약
✔ 1. 데이터 로드 성공
	•	JSON 3개 파일 정상 로드
	•	구조/형식 에러 없음

✔ 2. 데이터 스케일 파악
	•	고객 수 (17,000)
	•	오퍼 수 (10)
	•	이벤트 로그 (30만+)

✔ 3. 테이블 역할 정의
	•	transcript = 중심
	•	profile / portfolio = 보조 설명 테이블

✔ 4. 퍼널 정의 가능성 검증
	•	event 종류 확인
	•	라이프사이클 퍼널 구조 증명

✔ 5. value 컬럼 구조 이해
	•	이벤트별 스키마 차이 확인
	•	offer id / offer_id 혼재 발견

✔ 6. 조인 키 무결성 확인
	•	id 유일성 / 결측 없음

In [9]:
import pandas as pd

In [10]:
#1.데이터 로드

profile = pd.read_json('../data/raw/profile.json', orient='records', lines=True)
portfolio = pd.read_json('../data/raw/portfolio.json', orient='records', lines=True)
transcript = pd.read_json('../data/raw/transcript.json', orient='records', lines=True)

In [11]:
#2.데이터 규모 /컬럼 확인 
print("Profile shape:", profile.shape)
print("Portfolio shape:", portfolio.shape)
print("Transcript shape:", transcript.shape)

print("\nProfile columns:")
print(profile.columns.tolist())

print("\nPortfolio columns:")
print(portfolio.columns.tolist())

print("\nTranscript columns:")
print(transcript.columns.tolist())

Profile shape: (17000, 5)
Portfolio shape: (10, 6)
Transcript shape: (306534, 4)

Profile columns:
['gender', 'age', 'id', 'became_member_on', 'income']

Portfolio columns:
['reward', 'channels', 'difficulty', 'duration', 'offer_type', 'id']

Transcript columns:
['person', 'event', 'value', 'time']


profile : 고객 17,000명, porfolio : 오퍼 단위 10종, transcript : 306,534 고객행동 이벤트 로그 (30만건 이상))

In [12]:
# 상위 3개 미리 보기 (너무 많이 보면 산만해짐)
display(profile.head(3))
display(portfolio.head(3))
display(transcript.head(3))

,gender,age,id,became_member_on,income
0,None,118,68be06ca386d4c31939f3a4f0e3dd783,20170212,NaN
1,F,55,0610b486422d4921ae7d2bf64640c50b,20170715,112000.0
2,None,118,38fe809add3b4fcf9315a9694bb96ff5,20180712,NaN


,reward,channels,difficulty,duration,offer_type,id
0,10,"[email, mobile, social]",10,7,bogo,ae264e3637204a6fb9bb56bc8210ddfd
1,10,"[web, email, mobile, social]",10,5,bogo,4d5c57ea9a6940dd891ad53e9dbe8da0
2,0,"[web, email, mobile]",0,4,informational,3f207df678b143eea3cee63160fa8bed


,person,event,value,time
0,78afa995795e4d85b5d9ceeca43f5fef,offer received,{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},0
1,a03223e636434f42ac4c3df47e8bac43,offer received,{'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},0
2,e2127556f4f64592b11af22de27a7932,offer received,{'offer id': '2906b810c7d4411798c6938adc9daaa5'},0


In [13]:
# 퍼널의 단계가 되는 event가 실제로 어떤 값들인지 확인
transcript['event'].value_counts()

event
transaction        138953
offer received      76277
offer viewed        57725
offer completed     33579
Name: count, dtype: int64

	•	offer received → viewed → completed → transaction<br>
→ 이벤트 단계가 정확히 퍼널 형태<br>
	•	viewed 대비 completed 비율이 급격히 감소<br>
→ 퍼널 병목 후보 구간<br>
	•	transaction이 가장 많음<br>
→ 오퍼 없이도 결제하는 고객 존재 가능성 있음 (중요한 인사이트 씨앗)<br>

In [14]:
# value 컬럼은 dict(사전)일 가능성이 높음 → 샘플로 구조 확인
transcript['value'].head(10).tolist()

[{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'},
 {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},
 {'offer id': '2906b810c7d4411798c6938adc9daaa5'},
 {'offer id': 'fafdcd668e3743c1bb461111dcafc2a4'},
 {'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'},
 {'offer id': 'f19421c1d4aa40978ebb69ca19b0e20d'},
 {'offer id': '2298d6c36e964ae4a3e7e9706d1fb8c2'},
 {'offer id': '3f207df678b143eea3cee63160fa8bed'},
 {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'},
 {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}]

In [15]:
# 이벤트별 value 예시 보기
for e in transcript['event'].unique():
    sample = transcript.loc[transcript['event'] == e, 'value'].head(3).tolist()
    print(f"\n[{e}] value sample:")
    print(sample)
    


[offer received] value sample:
[{'offer id': '9b98b8c7a33c4b65b9aebfe6a799e6d9'}, {'offer id': '0b1e1539f2cc45b7b9fa7c272da2e1d7'}, {'offer id': '2906b810c7d4411798c6938adc9daaa5'}]

[offer viewed] value sample:
[{'offer id': 'f19421c1d4aa40978ebb69ca19b0e20d'}, {'offer id': '5a8bc65990b245e5a138643cd4eb9837'}, {'offer id': '4d5c57ea9a6940dd891ad53e9dbe8da0'}]

[transaction] value sample:
[{'amount': 0.8300000000000001}, {'amount': 34.56}, {'amount': 13.23}]

[offer completed] value sample:
[{'offer_id': '2906b810c7d4411798c6938adc9daaa5', 'reward': 2}, {'offer_id': 'fafdcd668e3743c1bb461111dcafc2a4', 'reward': 2}, {'offer_id': '9b98b8c7a33c4b65b9aebfe6a799e6d9', 'reward': 5}]


value 컬럼은 이벤트 유형에 따라 서로 다른 스키마를 가진다.
특히 오퍼 관련 이벤트에서는 'offer id'와 'offer_id'가 혼재되어 있어,
전처리 단계에서 이를 하나의 offer_id 컬럼으로 정규화하는 것이 필요해 보임

In [16]:
# 조인키가 정상인지 (중복, 결측) 점검

# profile: 고객 id 유일성 확인
print("profile id unique?:", profile['id'].is_unique)
print("profile id nulls:", profile['id'].isna().sum())

# portfolio: offer id 유일성 확인
print("portfolio id unique?:", portfolio['id'].is_unique)
print("portfolio id nulls:", portfolio['id'].isna().sum())

# transcript: person 결측 확인
print("transcript person nulls:", transcript['person'].isna().sum())

profile id unique?: True
profile id nulls: 0
portfolio id unique?: True
portfolio id nulls: 0
transcript person nulls: 0


## 데이터 이해 단계 수행 결과

- profile 테이블은 17,000명의 고객 정보를 담고 있으며,
  성별, 연령, 소득, 가입일 등의 인구통계 속성을 포함한다.
- portfolio 테이블은 총 10개의 오퍼 메타데이터로 구성되어 있으며,
  오퍼 유형, 보상, 조건 난이도, 유효기간, 전달 채널 정보를 제공한다.
- transcript 테이블은 약 30만 건의 고객 행동 이벤트 로그로 구성되어 있으며,
  고객(person), 이벤트 유형(event), 시간(time), 상세 정보(value)를 포함한다.

**본 분석은 transcript를 중심으로,
고객(profile) 및 오퍼(portfolio) 정보를 결합하여
오퍼 라이프사이클 기반 퍼널 및 이탈 분석을 수행한다.**

## **분석 내용 흐름 (가설)**
- 스타벅스 앱에서 고객은 오퍼를 받지만, 모두가 반응하진 않음
- 어디서 반응이 끊기는지?(퍼널 분석) 누가 이탈 위험(이탈분석)이 높은지? 
- 그렇다면 누구(고객 세그먼트 기준)에게 어떤 오퍼를 언제 제안해야 하는 것이 좋을까? : 데이터 기반 프로덕트(오퍼 서비스)를 개선 전략 도출